In [1]:
from bokeh.plotting import figure, show, output_file, save
from bokeh.io import output_notebook
import pandas as pd
from bokeh.models import ColumnDataSource, DataTable, TableColumn, HoverTool, StringFormatter, NumberFormatter, Div
from bokeh.layouts import column, row

In [2]:
output_notebook()

Loading BokehJS ...

In [21]:
df = pd.read_csv('../data/inst_tab.csv')

In [22]:
df.head()

,inst_name,ror_id,n_kb,n_oal,difference
0,Carl von Ossietzky Universität Oldenburg,https://ror.org/033n9gh91,5906,6452,-546
1,Georg-August-Universität Göttingen,https://ror.org/01y9bpm73,19881,21917,-2036
2,Gottfried Wilhelm Leibniz Universität Hannover,https://ror.org/0304hq317,11399,11768,-369
3,Hochschule Emden/Leer,https://ror.org/01bc76c69,292,345,-53
4,Hochschule Hannover,https://ror.org/03z6vda50,87,272,-185


In [23]:
df.replace({'Hochschule für angewandte Wissenschaft und Kunst Hildesheim/Holzminden/Göttingen': 'HAWK Hochschule für angewandte Wissenschaft und Kunst'}, inplace=True)

In [24]:
df.loc[len(df)] = ['Norddeutsche Hochschule für Rechtspflege', 'https://ror.org/02743t710', 0, 1, -1]

In [25]:
df.sort_values(by='inst_name', ascending=True, inplace=True)

In [26]:
source = ColumnDataSource(data=df)

In [27]:
TOOLS = ['hover']

TOOLTIPS = [
    ('Institution', '@inst_name'),
    ('Unterschied zu OpenAlex', '@difference')
]

p = figure(x_range=(df.difference.min()*1.2, df.difference.max()*1.2), 
           y_range=df.inst_name.tolist(),
           height=600, 
           width=700,
           title='Vergleich Institutionszuordnung in OPENBIB und OpenAlex',
           toolbar_location=None, 
           tools=TOOLS,
           tooltips=TOOLTIPS
          )

p.hbar(y='inst_name', 
       right='difference', 
       height=0.7, 
       hover_color='#c3c3c3', 
       color='#56B4E9',
       source=source)

p.ygrid.grid_line_color = None

show(p)

In [28]:
output_file(filename='comparison.html', title='Vergleich Institutionszuordnung in OPENBIB und OpenAlex')

columns = [
    TableColumn(field='inst_name', title='Institution'),
    TableColumn(field='ror_id', title='ROR ID'),
    TableColumn(field='n_kb', 
                title='Anzahl Items in OPENBIB',
                formatter=NumberFormatter(format='0,0')
               ),
    TableColumn(field='n_oal', 
                title='Anzahl Items in OpenAlex',
                formatter=NumberFormatter(format='0,0')
               )
]

data_table = DataTable(source=source, 
                       columns=columns, 
                       editable=False, 
                       row_height=35,
                       width=1200,
                       height=750
                      )

TOOLS = ['hover', 'reset']

TOOLTIPS = [
    ('Institution', '@inst_name'),
    ('Anzahl Items in OpenAlex', '@n_oal'),
    ('Anzahl Items in OPENBIB', '@n_kb')
]

p_1 = figure(width=450, 
             height=600,
             #title='Vergleich Institutionszuordnung in OPENBIB und OpenAlex',
             toolbar_location=None 
            )

p_11 = p_1.scatter(x='n_kb',
                 y='n_oal',
                 fill_color='#56B4E9', 
                 size=11, 
                 alpha=0.7,
                 source=source)

p_1.xaxis.axis_label = 'Anzahl der Publikationen in OPENBIB'
p_1.yaxis.axis_label = 'Anzahl der Publikationen in OpenAlex'

x = [x for x in range(0, int(df.n_oal.max() * 1.25), 1000)]
y = x

p_21 = p_1.line(x=x, 
              y=y, 
              color='#c3c3c3',
              line_dash='dashed',
              line_width=2,
              alpha=0.5
             )

p_1.add_tools(HoverTool(tooltips=TOOLTIPS, renderers=[p_11]))

TOOLS_p2 = ['hover']

TOOLTIPS_p2 = [
    ('Institution', '@inst_name'),
    ('Unterschied zu OpenAlex', '@difference')
]

p_2 = figure(x_range=(df.difference.min()*1.2, df.difference.max()*1.2), 
           y_range=df.inst_name.tolist(),
           height=600, 
           width=750,
           #title='Vergleich Institutionszuordnung in OPENBIB und OpenAlex',
           toolbar_location=None, 
           tools=TOOLS_p2,
           tooltips=TOOLTIPS_p2
          )

p_21 = p_2.hbar(y='inst_name', 
       right='difference', 
       height=0.7, 
       hover_color='#c3c3c3', 
       color='#56B4E9',
       source=source)

p_2.xaxis.axis_label = 'Unterschied zu OpenAlex (Anzahl der Publikationen)'
p_2.yaxis.axis_label = 'Einrichtung'

p_2.ygrid.grid_line_color = None

div_header = Div(text="""
                      <h1 style=font-size:16px>Vergleich der Institutionszuordnung niedersächsischer Hochschulen in OpenAlex und OPENBIB</h1>
                      """
                )

div_text = Div(text="""
                    <p>Die folgende Grafik vergleicht die Zuordnung von wissenschaftlichen Publikationen zu niedersächsischen Hochschulen in OpenAlex und OPENBIB.<br>
                    Es werden nur Publikationen berücksichtigt, die eine Zuordnung zu einer niedersächsischen Hochschule haben und zwischen 2020 und 2024 erschienen sind.</p>
                    <br>
                    <p><b>Datenquellen:</b></p>
                    <ul>
                        <li><b>OpenAlex:</b> Stand März 2026</li>
                        <li><b>OPENBIB:</b> Stand Juli 2025</li>
                    </ul>
                    """
                )

layout = column(
            column(div_header, div_text), 
            column(row(p_1, p_2), data_table, spacing=25)
)

show(layout)
save(layout)

'/Users/naustica/Desktop/lower_saxony_institutions/notebooks/comparison.html'